In [ ]:
#9. Documentation and Data Dictionary

In [1]:
#Ingest all sheets with pandas.read_excel().

import pandas as pd
import numpy as np
import os
from datetime import datetime

# Load all CSVs
customers = pd.read_csv("Customers.csv")
models = pd.read_csv("MobileModels.csv")
shops = pd.read_csv("ShopLocations.csv")
sales = pd.read_csv("Sales.csv")
stocks = pd.read_csv("Stocks.csv")
offers = pd.read_csv("Offers.csv")

# Store in a dictionary for easy iteration
datasets = {
    "Customers": customers,
    "MobileModels": models,
    "ShopLocations": shops,
    "Sales": sales,
    "Stocks": stocks,
    "Offers": offers
}

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [2]:
#Generate an automated Data Dictionary (columns, types, example values, simple definitions)

# 1_data_dictionary.py
import pandas as pd
import numpy as np
import os

# Edit this dictionary if any variable names differ
datasets = {
    "customers": customers,
    "mobilemodels": models,   # change key name if your df variable is named differently
    "shops": shops,
    "sales": sales,
    "offers": offers,
    "stocks": stocks
}

out_rows = []

def short_sample(series, n=3):
    vals = series.dropna().unique()[:n]
    return ", ".join([str(v) for v in vals])

# Basic heuristic for suggested short definition (very small)
def suggest_definition(col_name):
    cn = col_name.lower()
    if "id" in cn and len(cn) <= 10:
        return "Identifier (primary/foreign key)"
    if "date" in cn or "time" in cn:
        return "Date/time field"
    if any(k in cn for k in ["price", "amount", "revenue", "final", "base"]):
        return "Monetary amount (INR)"
    if "discount" in cn:
        return "Discount percentage or amount"
    if any(k in cn for k in ["qty", "quantity", "units"]):
        return "Quantity (units sold)"
    if "email" in cn:
        return "Customer email address"
    if "phone" in cn or "mobile" in cn:
        return "Phone number"
    if any(k in cn for k in ["city","location","shop","store"]):
        return "Geographic or shop location"
    if "rating" in cn:
        return "Numeric rating (e.g., shop rating)"
    if "loyalty" in cn:
        return "Customer loyalty points"
    return ""

for ds_name, df in datasets.items():
    if df is None:
        continue
    for col in df.columns:
        series = df[col]
        dtype = str(series.dtype)
        missing_pct = round(series.isnull().mean() * 100, 3)
        unique_count = series.nunique(dropna=True)
        sample_vals = short_sample(series)
        definition_guess = suggest_definition(col)
        out_rows.append({
            "dataset": ds_name,
            "column": col,
            "dtype": dtype,
            "missing_pct": missing_pct,
            "unique_count": unique_count,
            "sample_values": sample_vals,
            "suggested_definition": definition_guess,
            "final_definition": ""  # place for you to write the human definition
        })

data_dict = pd.DataFrame(out_rows)
data_dict = data_dict[["dataset","column","dtype","missing_pct","unique_count","sample_values","suggested_definition","final_definition"]]

# Save files
os.makedirs("documentation", exist_ok=True)
data_dict.to_csv("documentation/data_dictionary.csv", index=False)
data_dict.to_excel("documentation/data_dictionary.xlsx", index=False)

print("Saved data dictionary to documentation/data_dictionary.csv and .xlsx")
data_dict.head(20)

Saved data dictionary to documentation/data_dictionary.csv and .xlsx


,dataset,column,dtype,missing_pct,unique_count,sample_values,suggested_definition,final_definition
0,customers,customer_id,object,0.000,6000,"CUST1000, CUST1001, CUST1002",,
1,customers,full_name,object,0.000,454,"Arjun Sharma, Ishita Nair, Rohan Sharma",,
2,customers,email,object,5.183,5585,"ishita.shah834@gmail.com, ishita.kulkarni493@g...",Customer email address,
3,customers,phone,float64,3.900,5766,"5157281880.0, 1790139012.0, 203432336.0",Phone number,
4,customers,city,object,0.000,10,"Suratt, Delhi, Ahmedbad",Geographic or shop location,
5,customers,loyalty_points,float64,10.067,357,"138.0, 130.0, 101.0",Customer loyalty points,
6,mobilemodels,model_id,object,0.000,200,"MOD2000, MOD2001, MOD2002",Identifier (primary/foreign key),
7,mobilemodels,brand,object,0.000,8,"Oppo, OnePlus, Xiaomi",,
8,mobilemodels,model_name,object,0.000,20,"Series19, Series6, Series16",,
9,mobilemodels,ram_gb,int64,0.000,4,"12, 6, 4",,


In [3]:
#TRANSFORMATION LOG — Before → After Tracking
import pandas as pd
transformations = []
# Fixing city spellings
before_cities = customers['city'].unique().tolist()

city_corrections = {
    "Kolkatta": "Kolkata",
    "Hydrabad": "Hyderabad",
    "Bombay": "Mumbai",
    "Delhii": "Delhi",
    "Bengluru": "Bangalore",
    "Suratt": "Surat",
    "Ahmedbad": "Ahmedabad"
}
customers['city'] = customers['city'].replace(city_corrections)
after_cities = customers['city'].unique().tolist()
transformations.append({
    "step": "Fix city spellings",
    "column": "city",
    "before": before_cities,
    "after": after_cities
})
#Filling missing values
before_nulls = customers['email'].isna().sum()
customers['email'].fillna("unknown@example.com", inplace=True)
after_nulls = customers['email'].isna().sum()

transformations.append({
    "step": "Fill missing emails",
    "column": "email",
    "before": f"{before_nulls} nulls",
    "after": f"{after_nulls} nulls"
})
# Convert log to DataFrame and save
log_df = pd.DataFrame(transformations)
log_df.to_csv("Transformation_Log.csv", index=False)

print("Transformation Log generated!")

Transformation Log generated!


C:\Users\TANU\AppData\Local\Temp\ipykernel_9212\1914862292.py:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  customers['email'].fillna("unknown@example.com", inplace=True)


In [4]:
#PROFILING SUMMARY — Nulls, Duplicates, Outliers

import pandas as pd
import numpy as np
import os

folder_path = r"C:/Users/TANU/Documents/EDAProject"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

profile_rows = []

for file in csv_files:
    df = pd.read_csv(os.path.join(folder_path, file))

    # Null % per column
    null_pct = df.isna().mean().mean() * 100
    
    # Duplicate rows
    dup_count = df.duplicated().sum()

    # Outliers: numeric columns only
    numeric_cols = df.select_dtypes(include=['int64', 'float64'])
    outlier_count = 0

    for col in numeric_cols:
        Q1 = numeric_cols[col].quantile(0.25)
        Q3 = numeric_cols[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outlier_count += numeric_cols[(numeric_cols[col] < lower) | (numeric_cols[col] > upper)].shape[0]

    profile_rows.append({
        "table": file.replace(".csv", ""),
        "rows": df.shape[0],
        "columns": df.shape[1],
        "avg_missing_%": round(null_pct, 2),
        "duplicate_rows": dup_count,
        "numeric_outliers": outlier_count
    })

profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv("Profiling_Summary.csv", index=False)

print(profile_df)
print("Profiling summary saved!")

                         table   rows  columns  avg_missing_%  duplicate_rows  \
0                    Customers   6000        6           2.99               0   
1   Customers_with_names_split   6000        8           2.24               0   
2               DataDictionary      5        1           0.00               0   
3        Data_Metadata_Summary      6        7           0.00               0   
4                 MobileModels    200        6           0.00               0   
5           MobileModels_Fixed    200        6           0.00               0   
6    models_avg_price_by_brand      8        2           0.00               0   
7                       Offers    500        7           0.00               0   
8                        Sales  40000       15           0.00               0   
9                ShopLocations    100        4           0.00               0   
10                      Stocks   8000        5           0.00               0   

    numeric_outliers  
0   

In [5]:
#Create a README.md programmatically that summarizes project workflow

# 4_generate_readme.py
from datetime import date
import os

project_name = "MobileShoppee Data Curation & Analytics"
author = "Tanashree"
today = date.today().isoformat()

readme = f"""# {project_name}

**Author:** {author}  
**Date:** {today}

## Project Overview
This project ingests and curates mobile shop datasets (Customers, MobileModels, Shops, Sales, Offers, Stocks),
performs data cleaning, profiling, integration (star schema), feature engineering, and predictive modeling (regression).

## Folder structure (generated)
- `documentation/` : data dictionary, transformation log, profiling summaries (CSV/MD/XLSX)
- `notebooks/` : Jupyter notebooks for ETL, EDA, modeling
- `data/` : raw and cleaned CSV files
- `models/` : trained model artifacts (if any)
- `reports/` : final report, figures, slides

## Workflow
1. Load raw CSVs from `data/` into pandas DataFrames.
2. Run data profiling and generate `documentation/profiling_summary.*`.
3. Clean and standardize columns (snake_case), fix misspellings, handle missing values.
4. Log every transformation with `documentation/transformation_log.csv`.
5. Join tables into star schema:
   - Fact: `FactSales` (sales + revenue)
   - Dims: `DimCustomer`, `DimModel`, `DimShop`, `DimOffer`, `DimStock`
6. Feature engineering: revenue, discount_value, stock_turnover, price buckets, loyalty tiers.
7. Train models (XGBoost recommended) to predict final_price or revenue.
8. Save models and evaluation metrics in `models/` and `reports/`.

## Important files produced
- `documentation/data_dictionary.csv` and `.xlsx`
- `documentation/transformation_log.csv`
- `documentation/profiling_summary.csv` and `.md`
- `customers_cleaned.csv`, `sales_cleaned.csv`, etc.

## How to re-run
1. Place raw CSVs in `data/` and update the `folder_path` variables in the notebooks.
2. Run the notebook cells in order:
   - 00_load_data.ipynb
   - 01_profile_and_backup.ipynb
   - 02_cleaning_and_transforms.ipynb
   - 03_integration_star_schema.ipynb
   - 04_modeling.ipynb
3. Use `TransformationLogger` to record every change.

## Notes
- Great Expectations is not used due to environment constraints; all validations implemented using pandas checks.
- Use `documentation/profiling_summary.md` for a quick review before modeling.

"""

os.makedirs("documentation", exist_ok=True)
with open("documentation/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README saved to documentation/README.md")

README saved to documentation/README.md
